In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
def remove_dc_offset(complex_matrix):
    """
    Subtracts the complex mean from the dataset.
    Ref: 'Dataset Augmentation.md'
    """
    # 1. Calculate global mean
    dc_bias = np.mean(complex_matrix)
    
    # 2. Subtract (Zero-centering)
    centered_matrix = complex_matrix - dc_bias
    
    print(f"   > DC Offset removed. Mean shifted from {dc_bias:.2e} to {np.mean(centered_matrix):.2e}")
    return centered_matrix

In [ ]:
def process_npy_file(file_path):
    print(f"--- Processing {file_path.name} ---")
    
    raw_data = np.load(file_path)
    
    # Apply Correction
    clean_data = remove_dc_offset(raw_data)
    
    # Save to Final Folder
    save_path = MIT_PROCESSED_DATA_DIR / (file_path.stem + "_clean.npy")
    np.save(save_path, clean_data)
    print(f"Saved to {save_path}")

In [ ]:
files = list(MIT_PROCESSED_DATA_DIR.glob("*.npy"))
for f in files:
    process_npy_file(f)